# AnimeGANv2 Face2Paint 演示 (全中文教学版)

本 Notebook 旨在演示 `animegan2-pytorch` 项目的核心功能。
**说明：** 请从上到下按顺序运行所有代码单元格。

In [ ]:
#@title 1. 加载 Face2Paint 模型
import torch
from PIL import Image

print("步骤 1：加载 AnimeGANv2 模型...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", device=device, progress=False).eval()
face2paint = torch.hub.load("bryandlee/animegan2-pytorch:main", "face2paint", device=device, progress=False, side_by_side=True)
print(f"模型加载成功，使用设备: {device}")

In [ ]:
#@title 2. 定义人脸检测与对齐函数

import os
import dlib
import collections
from typing import Union, List
import numpy as np
from PIL import Image
import PIL.ImageFile
import scipy.ndimage
import matplotlib.pyplot as plt

print("步骤 2：定义人脸检测与对齐的辅助函数...")

def get_dlib_face_detector(predictor_path: str = "shape_predictor_68_face_landmarks.dat"):
    if not os.path.isfile(predictor_path):
        print(f"关键点预测器 '{predictor_path}' 未找到，正在下载...")
        model_file = "shape_predictor_68_face_landmarks.dat.bz2"
        os.system(f"wget http://dlib.net/files/{model_file}")
        os.system(f"bzip2 -dk {model_file}")
        print("下载完成。")
    detector = dlib.get_frontal_face_detector()
    shape_predictor = dlib.shape_predictor(predictor_path)
    def detect_face_landmarks(img: Union[Image.Image, np.ndarray]) -> List[np.ndarray]:
        if isinstance(img, Image.Image):
            img = np.array(img)
        faces = []
        dets = detector(img, 1)
        for d in dets:
            shape = shape_predictor(img, d)
            faces.append(np.array([[v.x, v.y] for v in shape.parts()]))
        return faces
    return detect_face_landmarks

def display_facial_landmarks(img: Image.Image, landmarks: List[np.ndarray], fig_size=[10, 10]):
    plot_style = {'marker': 'o', 'markersize': 4, 'linestyle': '-', 'lw': 2}
    pred_type = collections.namedtuple('prediction_type', ['slice', 'color'])
    pred_types = {
        'face': pred_type(slice(0, 17), (0.682, 0.780, 0.909, 0.5)),
        'eyebrow1': pred_type(slice(17, 22), (1.0, 0.498, 0.055, 0.4)),
        'eyebrow2': pred_type(slice(22, 27), (1.0, 0.498, 0.055, 0.4)),
        'nose': pred_type(slice(27, 31), (0.345, 0.239, 0.443, 0.4)),
        'nostril': pred_type(slice(31, 36), (0.345, 0.239, 0.443, 0.4)),
        'eye1': pred_type(slice(36, 42), (0.596, 0.875, 0.541, 0.3)),
        'eye2': pred_type(slice(42, 48), (0.596, 0.875, 0.541, 0.3)),
        'lips': pred_type(slice(48, 60), (0.596, 0.875, 0.541, 0.3)),
        'teeth': pred_type(slice(60, 68), (0.596, 0.875, 0.541, 0.4))
    }
    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(1, 1, 1)
    ax.imshow(img)
    ax.axis('off')
    for face in landmarks:
        for pt_type in pred_types.values():
            ax.plot(face[pt_type.slice, 0], face[pt_type.slice, 1], color=pt_type.color, **plot_style)
    plt.show()

def align_and_crop_face(
    img: Image.Image,
    landmarks: np.ndarray,
    expand: float = 1.0,
    output_size: int = 1024,
    transform_size: int = 4096,
    enable_padding: bool = True
):
    lm = landmarks
    lm_eye_left      = lm[36 : 42]
    lm_eye_right     = lm[42 : 48]
    lm_mouth_outer   = lm[48 : 60]
    eye_left    = np.mean(lm_eye_left, axis=0)
    eye_right   = np.mean(lm_eye_right, axis=0)
    eye_avg     = (eye_left + eye_right) * 0.5
    eye_to_eye  = eye_right - eye_left
    mouth_left  = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg   = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg
    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    x *= expand
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2
    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, Image.Resampling.LANCZOS)
        quad /= shrink
        qsize /= shrink
    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0), min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]
    pad = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0), max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img_np = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img_np.shape
        y, x, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(1.0 - np.minimum(np.float32(x) / pad[0], np.float32(w-1-x) / pad[2]), 1.0 - np.minimum(np.float32(y) / pad[1], np.float32(h-1-y) / pad[3]))
        blur = qsize * 0.02
        img_np += (scipy.ndimage.gaussian_filter(img_np, [blur, blur, 0]) - img_np) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img_np += (np.median(img_np, axis=(0,1)) - img_np) * np.clip(mask, 0.0, 1.0)
        img = Image.fromarray(np.uint8(np.clip(np.rint(img_np), 0, 255)), 'RGB')
        quad += pad[:2]
    img = img.transform((transform_size, transform_size), Image.Transform.QUAD, (quad + 0.5).flatten(), Image.Resampling.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), Image.Resampling.LANCZOS)
    return img

print("辅助函数定义成功。")

In [ ]:
#@title 3. 处理图像并获取结果
import requests
from io import BytesIO
from PIL import Image

img_url = "https://this-person-does-not-exist.com/img/avatar-gen1121c2567427214741399d863f63b22b.jpg"

print(f"步骤 3：处理来自 URL 的图像: {img_url}")
try:
    response = requests.get(img_url)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB")
    face_detector = get_dlib_face_detector()
    landmarks = face_detector(img)

    if landmarks:
        print(f"检测到 {len(landmarks)} 张人脸。正在处理第一张...")
        face = align_and_crop_face(img, landmarks[0], output_size=512)
        print("人脸已对齐。正在生成动漫风格图像...")
        # 在 Jupyter 中，单元格的最后一个表达式会自动显示其内容。
        # 我们使用 display() 函数以确保在所有环境中都能显式地渲染图像。
        display(face2paint(model=model, img=face))
    else:
        print("\n错误：未在图像中检测到任何人脸。请尝试另一张图片。")
        display(img)

except requests.exceptions.RequestException as e:
    print(f"\n错误：无法下载图像。{e}")
except Exception as e:
    print(f"\n错误：处理过程中发生意外。{e}")